# Specify the State and Data File Paths

In [1]:
# Gecorr 2022 Census Block Group to PUMA
geocorr_dict_path = "./data/geocorr/geocorr2022_CA_CensusBlockGroup_to_PUMA.csv"

# Social Vulnerability Index (SVI)
svi_csv_file_path = "./data/social_vulnerability_index/SVI2022_CA_CensusTract.csv"

# Neighborhood Atlas (ADI)
adi_csv_file_path = (
    "./data/neighborhood_atlas/CA_2022_ADI_Census_Block_Group_v4_0_1.csv"
)

# Public Use Microdata Sample (PUMS)
state = "CA"
year = 2023

# Obtain FIPS Codes for Census Tract or Census Block Group

- For [Neighborhood Atlas](https://www.neighborhoodatlas.medicine.wisc.edu/), the geolocation is coded with a 12-digit FIPS code, which takes the form of SSCCCTTTTTTG (2-digit state, 3-digit county, 6-digit census tract, and 1-digit census block group).
- For [Social Vulnerability Index](https://www.atsdr.cdc.gov/place-health/php/svi/svi-data-documentation-download.html), the geolocation is coded with a 11-digit FIPS code, which takes the form of SSCCCTTTTTT (2-digit state, 3-digit county, 6-digit census tract).

In [2]:
from utils.geocorr_processing import (
    generate_geocorr_dictionary,
    calculate_weights,
    validate_geocorr_dict_structure,
)

import pandas as pd

geocorr_dict = generate_geocorr_dictionary(
    csv_file_path=geocorr_dict_path,
)

try:
    validate_geocorr_dict_structure(geocorr_dict)
    print("Dictionary structure is valid")
except ValueError as e:
    print(f"Invalid dictionary structure: {e}")

Dictionary generated with 281 PUMA keys
Saved to: None
Dictionary structure is valid


# Social Vulnerability Index (SVI)

- 11-digit FIPS code, which takes the form of SSCCCTTTTTT (2-digit state, 3-digit county, 6-digit census tract)
- use FIPS to look up the `RPL_THEMES` (the overall percentile ranking) value in the SVI data at the census tract level, valid values is in [0., 1.] 
- perform weighted aggregation according to population to obtain the PUMA-level SVI
- `puma_key` is 7-digit (2-digit state, 5-digit puma)

In [3]:
from utils.ses_index import calculate_puma_svi_quantiles

svi_df = pd.read_csv(svi_csv_file_path)[["FIPS", "RPL_THEMES"]]

# SVI calculated according the census tract level population
puma_tract_population_weights = calculate_weights(geocorr_dict, "census_tract_level")

puma_svi_quantiles = calculate_puma_svi_quantiles(
    puma_tract_population_weights, svi_df, specific_index="RPL_THEMES"
)

# Neighborhood Atlas

- 12-digit FIPS code, which takes the form of SSCCCTTTTTTG (2-digit state, 3-digit county, 6-digit census tract, 1-digit census block group)
- use FIPS to look up the `ADI_NATRANK`, national percentile rankings, at the block group level from 1 to 100
- `puma_key` is 7-digit (2-digit state, 5-digit puma)

In [4]:
from utils.ses_index import calculate_puma_adi_rankings

adi_df = pd.read_csv(adi_csv_file_path)[["FIPS", "ADI_NATRANK", "ADI_STATERNK"]]

# ADI calculated according the census block group level population
puma_block_group_population_weights = calculate_weights(
    geocorr_dict, "census_block_group_level"
)

puma_adi_rankings = calculate_puma_adi_rankings(
    puma_block_group_population_weights, adi_df, specific_index="ADI_NATRANK"
)

# Obtain Public-Use Microdata Sample (PUMS) Data

- utilize Folktables function, which makes use of US Census Bureau data download APIs
- combine `STATE` and `PUMA` to obtain the 7-digit `puma_key` (2-digit state, 5-digit puma)
- **enhance** the PUMS data with `ADI_NATRANK` [1, 100], and `SVI_RPL_THEMES` [0., 1.]

In [5]:
import numpy as np
from folktables import ACSDataSource


def load_acs_pums_person_data(
    nodes, *, year=2023, states=["CA"], download=True, addon_feature=None
):
    """
    Load folktables data, which contains recent US census data.
    """
    data_source = ACSDataSource(survey_year=year, horizon="1-Year", survey="person")
    acs_data = data_source.get_data(states=states, download=download)

    # Perform basic filter (mimic the one implemented in UCI Adult)
    # acs_data = adult_filter(acs_data)

    # Include add-on features, e.g., ['ADJINC', 'PWGTP']
    include = nodes.copy()
    if None is not addon_feature:
        include = nodes + list(addon_feature)

    df = acs_data[include]

    data_dict = df.dropna().to_dict(orient="list")

    n_samples = len(data_dict[next(iter(data_dict))])

    for node, value in data_dict.items():
        data_dict[node] = np.array(value).reshape(
            -1,
        )

    return data_dict, n_samples

In [6]:
nodes = [
    "AGEP",
    #  "COW",
    #  "SCH",
    #  "SCHG",
    #  "SCHL",
    "SEX",
    "PWGTP",
    "RAC1P",
    "PINCP",
    "PUMA",
    "STATE",
]

pums_data_dict, n_samples = load_acs_pums_person_data(nodes, year=year, states=[state])
print(f"State: {state}, year: {year}, number of samples: {n_samples}.")

State: CA, year: 2023, number of samples: 334276.


In [7]:
# Convert STATE and PUMA to zero-padded strings
pums_data_dict["STATE"] = [str(int(x)).zfill(2) for x in pums_data_dict["STATE"]]
pums_data_dict["PUMA"] = [str(int(x)).zfill(5) for x in pums_data_dict["PUMA"]]
pums_data_dict["puma_key"] = [
    pums_data_dict["STATE"][i] + pums_data_dict["PUMA"][i]
    for i in range(len(pums_data_dict["STATE"]))
]

In [8]:
for i in range(5):
    print(
        f"{pums_data_dict['STATE'][i]} + {pums_data_dict['PUMA'][i]} = {pums_data_dict['puma_key'][i]}"
    )

06 + 05924 = 0605924
06 + 08515 = 0608515
06 + 02500 = 0602500
06 + 05916 = 0605916
06 + 08106 = 0608106


In [9]:
# Enhance the PUMA key with SVI and ADI
pums_data_dict["SVI"] = np.zeros(n_samples)
pums_data_dict["ADI"] = np.zeros(n_samples)

for i in range(n_samples):
    puma_key = pums_data_dict["puma_key"][i]
    if puma_key in puma_svi_quantiles:
        pums_data_dict["SVI"][i] = puma_svi_quantiles[puma_key]
    else:
        pums_data_dict["SVI"][i] = np.nan

    if puma_key in puma_adi_rankings:
        pums_data_dict["ADI"][i] = puma_adi_rankings[puma_key]
    else:
        pums_data_dict["ADI"][i] = np.nan

In [10]:
df_pums_enhanced = pd.DataFrame.from_dict(pums_data_dict)

In [11]:
df_pums_enhanced.head()

,AGEP,SEX,PWGTP,RAC1P,PINCP,PUMA,STATE,puma_key,SVI,ADI
0,19,2,85,8,3600.0,05924,06,0605924,0.305349,6.375410
1,23,1,60,1,800.0,08515,06,0608515,0.211121,1.058337
2,35,1,27,2,0.0,02500,06,0602500,0.786554,57.063497
3,60,1,23,1,2700.0,05916,06,0605916,0.708187,24.347565
4,23,1,51,1,1800.0,08106,06,0608106,0.362305,2.381216
